In [1]:
"""cifar10 hp-tuning cnn model: interactive notebook experiment.

Legacy Hyperas experiment template. create_data supplies image or stored-feature arrays,
and create_model contains double-brace search expressions that Hyperas expands before
execution. These templates retain the historical helper signatures and optimizer options
written in their cells.

The search choices, best-score globals, trial counter, dataset/model paths, and fit
arguments below are its inputs. Local function docstrings explain each objective and
preprocessing branch. Outputs include trial losses, logs, and saved best models where
the objective explicitly writes them. Running cells may initialize GPU resources,
load/download data, train models, and overwrite configured artifact paths.
"""

from __future__ import print_function

import os


os.chdir("../")

In [1]:
from hyperopt import STATUS_OK, Trials, tpe
from hyperas.distributions import choice, uniform, loguniform
from hyperas import optim

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, metrics

from sklearn.model_selection import train_test_split

import numpy as np

from dataloader import load_cifar10
from model import get_compile_args, get_callbacks
from common.utils import (hyperas_path, i, init,
                        best_score, save_logs)

In [2]:
def create_data():
    """Load the CIFAR10 image splits used by the Hyperas CNN search.

    Calls the notebook init helper before the dataset loader. Data loading may
    download/cache images and initialize TensorFlow resources. The function has no
    selectable preprocessing mode; this trial family fixes preprocess="min-max" and
    verbose=0.

    Args:
        None.

    Returns:
        tuple[np.ndarray, ...]: (x_train, y_train, x_val, y_val, x_test, y_test), with
        min-max image inputs and sparse class IDs.
    """

    init()

    x_train, y_train, x_val, y_val, x_test, y_test = load_cifar10(preprocess="min-max", verbose=0)

    return x_train, y_train, x_val, y_val, x_test, y_test

In [3]:
def create_model(x_train, y_train, x_val, y_val, x_test, y_test):
    """Run one legacy Hyperas CNN trial for 10-class CIFAR classification.

    Hyperas replaces the double-brace choice expressions before calling this template.
    Choices control convolution groups, normalization, pooling, dropout, and optimizer.
    The first two convolution groups are unconditional; the three/four/five mode appends
    the groups written in its branch, so the name is not the total group count. Training
    uses batch size 128 and at most 100 epochs with validation callbacks. The objective
    updates notebook globals best_acc and i, appends trial logs, and may overwrite the
    best HDF5 model. Test arrays are unused.

    Args:
        x_train (np.ndarray): Training images with shape (N, 32, 32, 3), min-max
            preprocessed by create_data.
        y_train (np.ndarray): Training class IDs aligned with the first axis of the
            training inputs.
        x_val (np.ndarray): Validation images used by fit callbacks and final trial
            accuracy.
        y_val (np.ndarray): Validation class IDs used for the optimization score.
        x_test (np.ndarray): Held-out images accepted for the Hyperas data contract;
            unused in this objective.
        y_test (np.ndarray): Held-out class IDs accepted for the data contract; unused
            in this objective.

    Returns:
        dict[str, object]: loss is negative validation accuracy, status is STATUS_OK,
        and model is None; the trained model is saved separately only when it improves
        best_acc.
    """

    def conv_layer(kernels, filter_size, batch_norm, num=1):
        """Build a repeated convolution stage followed by spatial max pooling.

        Layer weights are created according to Keras build timing. The helper constructs a
        separate stage on each call; it does not share weights or compile the result.

        Args:
            kernels (int): Output filters in every convolution in this stage.
            filter_size (int | tuple[int, int]): Spatial kernel size for same-padded
                convolutions.
            batch_norm (str): The literal "yes" inserts BatchNormalization and disables
                convolution biases; other values keep biases and omit normalization.
            num (int): Number of Conv2D/ReLU repetitions before one same-padded 2x2 max
                pool. Defaults to ``1``.

        Returns:
            tf.keras.Sequential: New stage that preserves spatial dimensions within each
            convolution and reduces them to ceil(H/2), ceil(W/2) with same-padded stride-two
            max pooling.
        """

        model = models.Sequential()

        for _ in range(num):
            # Batch normalization disables convolution bias and adds normalization; other
            # choices retain bias.
            model.add(layers.Conv2D(kernels, filter_size, padding="same", use_bias=False if batch_norm=="yes" else True))
            # Batch normalization disables convolution bias and adds normalization; other
            # choices retain bias.
            model.add(layers.BatchNormalization()) if batch_norm=="yes" else None
            model.add(layers.Activation("relu"))

        model.add(layers.MaxPooling2D(2, padding="same"))

        return model


    global best_acc, i


    conv_num = {{choice(["three", "four", "five"])}}
    first_conv_filter_size = {{choice([3, 5, 7, 9])}}
    batch_norm = {{choice(["yes", "no"])}}
    global_pooling_type = {{choice(["max", "avg"])}}

    convgp1_kernels = {{choice([4, 8, 16, 32, 64])}}
    convgp2_kernels = convgp1_kernels * {{choice([1, 2])}}
    convgp3_kernels = convgp2_kernels * {{choice([1, 2])}}
    convgp4_kernels = convgp3_kernels * {{choice([1, 2])}}
    convgp5_kernels = convgp4_kernels * {{choice([1, 2])}}
    
    convgp2_num = {{choice([1, 2, 3])}}
    convgp3_num = {{choice([1, 2, 3])}}
    convgp4_num = {{choice([1, 2, 3])}}
    convgp5_num = {{choice([1, 2, 3])}}

    dense_dropout = {{choice([0.0, 0.05, 0.1, 0.15, 0.20, 0.25])}}

    lr = {{choice([0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1])}}
    momentum = {{choice([0.0, 0.01, 0.03, 0.1, 0.3, 0.5, 0.8, 0.9, 0.99])}}
    weight_decay = {{choice([0.0, 0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1])}}
    opt_choice = {{choice(["sgd", "rmsprop", "adam", "nadam", "adamax", "adamw"])}}


    model = models.Sequential(name="cifar10_cnn_model_01")

    model.add(layers.Input(shape=(32, 32, 3)))
    model.add(conv_layer(convgp1_kernels, first_conv_filter_size, batch_norm))
    model.add(conv_layer(convgp2_kernels, 3, batch_norm, convgp2_num))

    # Append the three-mode convolution groups after the two unconditional stages.
    if conv_num == "three":
        model.add(conv_layer(convgp2_kernels, 3, batch_norm, convgp2_num))
        model.add(conv_layer(convgp3_kernels, 3, batch_norm, convgp3_num))
        
    # Append the four-mode convolution groups after the two unconditional stages.
    elif conv_num == "four":
       model.add(conv_layer(convgp2_kernels, 3, batch_norm, convgp2_num))
       model.add(conv_layer(convgp3_kernels, 3, batch_norm, convgp3_num))
       model.add(conv_layer(convgp4_kernels, 3, batch_norm, convgp4_num))
            
    # Append the five-mode convolution groups after the two unconditional stages.
    elif conv_num == "five":
        model.add(conv_layer(convgp2_kernels, 3, batch_norm, convgp2_num))
        model.add(conv_layer(convgp3_kernels, 3, batch_norm, convgp3_num))
        model.add(conv_layer(convgp4_kernels, 3, batch_norm, convgp4_num))
        model.add(conv_layer(convgp5_kernels, 3, batch_norm, convgp5_num))

    # Reduce each channel by its spatial maximum for the max-pooling treatment.
    if global_pooling_type == "max":
        model.add(layers.GlobalMaxPooling2D())

    # Reduce each channel by spatial average for the average-pooling treatment.
    elif global_pooling_type == "avg":
        model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dropout(dense_dropout))
    model.add(layers.Dense(10, activation="softmax"))

    sgd = optimizers.SGD(learning_rate=lr, momentum=momentum, decay=weight_decay, nesterov=True)
    rmsprop = optimizers.RMSprop(learning_rate=lr, momentum=momentum, decay=weight_decay)
    adam = optimizers.Adam(learning_rate=lr, decay=weight_decay)
    nadam = optimizers.Nadam(learning_rate=lr, decay=weight_decay)
    adamax = optimizers.Adamax(learning_rate=lr, decay=weight_decay)
    adamw = optimizers.experimental.AdamW(learning_rate=lr, weight_decay=weight_decay)

    # Use the sgd optimizer for this sampled optimizer choice.
    if opt_choice == "sgd":
        optimizer = sgd
    # Use the rmsprop optimizer for this sampled optimizer choice.
    elif opt_choice == "rmsprop":
        optimizer = rmsprop
    # Use the adam optimizer for this sampled optimizer choice.
    elif opt_choice == "adam":
        optimizer = adam
    # Use the nadam optimizer for this sampled optimizer choice.
    elif opt_choice == "nadam":
        optimizer = nadam
    # Use the adamax optimizer for this sampled optimizer choice.
    elif opt_choice == "adamax":
        optimizer = adamax
    # Use the adamw optimizer for this sampled optimizer choice.
    elif opt_choice == "adamw":
        optimizer = adamw

    model.compile(**create_compile_args(optimizer=optimizer))


    history = model.fit(
        x_train, y_train,
        batch_size=128,
        epochs=100,
        validation_data=(x_val, y_val),
        callbacks=create_callbacks_list(min_delta=1e-2, verbose=0),
        verbose=0,
    ).history

    val_acc = metrics.SparseCategoricalAccuracy()(y_val, model.predict(x_val, verbose=0))

    # Save this classifier only when validation accuracy improves the previous best.
    if val_acc > best_acc:
        best_acc = val_acc
        model.save(os.path.join(hyperas_path, f"{model.name}.h5"))

    save_logs(model.name, i, val_acc, best_acc, 
            search_space=[conv_num, first_conv_filter_size, batch_norm, global_pooling_type, 
                        convgp1_kernels, convgp2_kernels, convgp3_kernels, convgp4_kernels, convgp5_kernels, 
                        convgp2_num, convgp3_num, convgp4_num, convgp5_num, dense_dropout, lr, momentum, 
                        weight_decay, opt_choice], 
            names=["conv_num", "first_conv_filter_size", "batch_norm", "global_pooling_type", 
                "convgp1_kernels", "convgp2_kernels", "convgp3_kernels", "convgp4_kernels", "convgp5_kernels", 
                "convgp2_num", "convgp3_num", "convgp4_num", "convgp5_num", "dense_dropout", "lr", "momentum", 
                "weight_decay", "opt_choice"], 
            where_to="file")

    i += 1


    return {"loss": -val_acc, "status": STATUS_OK, "model": None}

In [4]:
best_run, best_model = optim.minimize(
    model=create_model,
    data=create_data,
    algo=tpe.suggest,
    max_evals=50,
    trials=Trials(),
    notebook_name="notebooks/cifar10 hp-tuning cnn model"
)

>>> Imports:
#coding=utf-8

from __future__ import print_function

try:
    from hyperopt import STATUS_OK, Trials, tpe
except:
    pass

try:
    from hyperas.distributions import choice, uniform, loguniform
except:
    pass

try:
    from hyperas import optim
except:
    pass

try:
    from tensorflow.keras import layers, models, optimizers, metrics
except:
    pass

try:
    from tensorflow.keras.datasets import cifar10
except:
    pass

try:
    from sklearn.model_selection import train_test_split
except:
    pass

try:
    import numpy as np
except:
    pass

try:
    import os
except:
    pass

try:
    from utils import create_compile_args, create_callbacks_list, hyperas_path, i, best_acc, save_logs
except:
    pass

try:
    import tensorflow as tf
except:
    pass

try:
    from sklearn.metrics import classification_report
except:
    pass

>>> Hyperas search space:

def get_space():
    return {
        'conv_num': hp.choice('conv_num', ["three", "four", "five"]),
        'fi

In [5]:
best_run

{'batch_norm': 0,
 'conv_num': 0,
 'convgp1_kernels': 4,
 'convgp1_kernels_1': 1,
 'convgp1_kernels_2': 1,
 'convgp1_kernels_3': 1,
 'convgp1_kernels_4': 0,
 'convgp2_num': 1,
 'convgp2_num_1': 0,
 'convgp2_num_2': 1,
 'convgp2_num_3': 2,
 'dense_dropout': 3,
 'first_conv_filter_size': 2,
 'global_pooling_type': 1,
 'lr': 2,
 'momentum': 4,
 'opt_choice': 1,
 'weight_decay': 4}

In [6]:
from sklearn.metrics import classification_report


*_, x_test, y_test = create_data()
model = models.load_model(os.path.join(hyperas_path, "cifar10_cnn_model_01.h5"))
model.summary()

preds = model.predict(x_test)
print(classification_report(y_test, np.argmax(preds, axis=-1), digits=4))

Model: "cifar10_model_01"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 sequential_6 (Sequential)   (None, 16, 16, 64)        9664      
                                                                 
 sequential_7 (Sequential)   (None, 8, 8, 128)         222208    
                                                                 
 sequential_8 (Sequential)   (None, 4, 4, 128)         295936    
                                                                 
 sequential_9 (Sequential)   (None, 2, 2, 256)         295936    
                                                                 
 global_average_pooling2d_1   (None, 256)              0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dropout_1 (Dropout)         (None, 256)               0         
                                                  